In [2]:
# !pip install "apted==1.0.3"
# !pip install "Levenshtein"
# !pip install pytest

In [3]:
"""
Tree distance similarity metric taken from <https://github.com/ibm-aur-nlp/PubTabNet/blob/master/src/metric.py>
"""

import argparse
from collections import deque
from typing import List, Optional, Tuple

from apted import APTED, Config
from apted.helpers import Tree
from Levenshtein import distance
from lxml import etree, html
from lxml.html import HtmlElement


In [4]:


class TableTree(Tree):
    def __init__(
        self,
        tag: str,
        colspan: Optional[int] = None,
        rowspan: Optional[int] = None,
        content: Optional[List[str]] = None,
        *children: Tuple[Tree],
    ):
        self.tag = tag
        self.colspan = colspan
        self.rowspan = rowspan
        self.content = content
        self.children = list(children)



class CustomConfig(Config):
    @staticmethod
    def maximum(*sequences):
        """Get maximum possible value"""
        return max(map(len, sequences))

    def normalized_distance(self, *sequences):
        """Get distance from 0 to 1"""
        return float(distance(*sequences)) / self.maximum(*sequences)

    def rename(self, node1: TableTree, node2: TableTree) -> float:
        """Compares attributes of trees"""
        if (
            (node1.tag != node2.tag)
            or (node1.colspan != node2.colspan)
            or (node1.rowspan != node2.rowspan)
        ):
            return 1.0

        if node1.tag == "td":
            if node1.content or node2.content:
                return self.normalized_distance(node1.content, node2.content)

        return 0.0





class TEDS_HTML:
    """Tree Edit Distance basead Similarity"""

    def __init__(
        self, structure_only: bool = False, ignore_nodes: Optional[str] = None
    ):
        self.structure_only = structure_only
        self.ignore_nodes = ignore_nodes
        self.__tokens__: List[str] = []

    def __call__(self, pred: str, gt: str) -> float:
        """Computes TEDS score between the prediction and the ground truth of a
        given sample

        Args:
            pred (str): The predict html string of the table image.
            gt (str): The ground truth html string of the table image.

        Returns:
            float: TEDS score
        """
        if (not pred) or (not gt):
            return 0.0

        parser = html.HTMLParser(remove_comments=True, encoding="utf-8")
        pred_element: HtmlElement = html.fromstring(pred, parser=parser)
        gt_element: HtmlElement = html.fromstring(gt, parser=parser)

        xpath_ele = "body/table"
        if pred_element.xpath(xpath_ele) and gt_element.xpath(xpath_ele):
            pred_element = pred_element.xpath(xpath_ele)[0]
            gt_element = gt_element.xpath(xpath_ele)[0]

 
            if self.ignore_nodes:
                etree.strip_tags(pred_element, *self.ignore_nodes)
                etree.strip_tags(gt_element, *self.ignore_nodes)

            tree_pred = self.load_html_tree(pred_element)
            tree_true = self.load_html_tree(gt_element)
            distance = APTED(
                tree_pred, tree_true, CustomConfig()
            ).compute_edit_distance()
            n_nodes_pred = len(pred_element.xpath(".//*"))
            n_nodes_true = len(gt_element.xpath(".//*"))
            n_nodes = max(n_nodes_pred, n_nodes_true)
            return 1.0 - (float(distance) / n_nodes)
        return 0.0

    def tokenize(self, node: HtmlElement):
        """Tokenizes table cells"""
        self.__tokens__.append(f"<{node.tag}>")

        if node.text is not None:
            self.__tokens__ += list(node.text)

        for n in node.getchildren():
            self.tokenize(n)

        if node.tag != "unk":
            self.__tokens__.append(f"</{node.tag}>")

        if node.tag != "td" and node.tail is not None:
            self.__tokens__ += list(node.tail)

    def load_html_tree(
        self, node: HtmlElement, parent: Optional[HtmlElement] = None
    ) -> Optional[TableTree]:
        """Converts HTML tree to the format required by apted"""
        global __tokens__
        if node.tag == "td":
            if self.structure_only:
                cell = []
            else:
                self.__tokens__ = []
                self.tokenize(node)
                cell = self.__tokens__[1:-1].copy()
            new_node = TableTree(
                node.tag,
                int(node.attrib.get("colspan", "1")),
                int(node.attrib.get("rowspan", "1")),
                cell,
                *deque(),
            )
        else:
            new_node = TableTree(node.tag, None, None, None, *deque())

        if parent is not None:
            parent.children.append(new_node)

        if node.tag != "td":
            for n in node.getchildren():
                self.load_html_tree(n, new_node)

        if parent is None:
            return new_node
        return None


In [5]:
from typing import Optional, List, Tuple, Dict
from collections import deque

class TableTree(Tree):
    def __init__(
        self,
        tag: str,
        colspan: Optional[int] = None,
        rowspan: Optional[int] = None,
        content: Optional[List[str]] = None,
        *children: Tuple['TableTree'],
    ):
        self.tag = tag
        self.colspan = colspan
        self.rowspan = rowspan
        self.content = content
        self.children = list(children)

class CustomConfigJson(Config):
    @staticmethod
    def maximum(*sequences):
        """Get maximum possible value"""
        return max(map(len, sequences))

    def normalized_distance(self, *sequences):
        """Get distance from 0 to 1"""
        return float(distance(*sequences)) / self.maximum(*sequences)

    def rename(self, node1: TableTree, node2: TableTree) -> float:
        """Compares attributes of trees"""
        if (
            (node1.tag != node2.tag)
            or (node1.colspan != node2.colspan)
            or (node1.rowspan != node2.rowspan)
        ):
            return 1.0

        if node1.tag == "cell":
            if node1.content or node2.content:
                return self.normalized_distance(node1.content, node2.content)

        return 0.0



class TEDS_JSON:
    """Tree Edit Distance based Similarity"""

    def __init__(
        self, structure_only: bool = False, ignore_nodes: Optional[List[str]] = None
    ):
        self.structure_only = structure_only
        self.ignore_nodes = ignore_nodes
        self.__tokens__: List[str] = []

    # print tokens
    def __str__(self):
        return str(self.__tokens__)

    def __call__(self, pred: Dict, gt: Dict) -> float:
        """Computes TEDS score between the prediction and the ground truth of a
        given sample

        Args:
            pred (Dict): The predicted JSON representation of the table.
            gt (Dict): The ground truth JSON representation of the table.

        Returns:
            float: TEDS score
        """
        if not pred or not gt:
            return 0.0

        tree_pred = self.load_json_tree(pred)

        tree_true = self.load_json_tree(gt)
        distance = APTED(tree_pred, tree_true, CustomConfigJson()).compute_edit_distance()
        n_nodes_pred = self.count_nodes(tree_pred) - 1 # -1 to exclude the root node
        n_nodes_true = self.count_nodes(tree_true) - 1 
        n_nodes = max(n_nodes_pred, n_nodes_true)
        return 1.0 - (float(distance) / n_nodes)

    def tokenize(self, node: Dict):
        """Tokenizes table cells"""
        self.__tokens__.append(f"<{node['type']}>")

        if 'value' in node and node['value']:
            self.__tokens__ += list(node['value'])

        if 'children' in node:
            for child in node['children']:
                self.tokenize(child)

        if node['type'] != "unk":
            self.__tokens__.append(f"</{node['type']}>")

    def load_json_tree(
        self, node: Dict, parent: Optional[TableTree] = None
    ) -> Optional[TableTree]:
        """Converts JSON tree to the format required by APTED"""
        if node['type'] == 'cell':
            if self.structure_only:
                cell_content = []
            else:
                self.__tokens__ = []
                self.tokenize(node)
                cell_content = self.__tokens__[1:-1].copy()

            new_node = TableTree(
                'cell',
                int(node.get('colspan', 1)),
                int(node.get('rowspan', 1)),
                cell_content,
                *deque(),
            )
        else: # table or row
            new_node = TableTree(
                node['type'],
                None,
                None,
                None,
                *deque(),
            )

        if parent is not None:
            parent.children.append(new_node)

        if 'children' in node: 
            for child in node['children']:
                self.load_json_tree(child, new_node)
        if parent is None:
            return new_node
        return None

    def count_nodes(self, tree: TableTree) -> int:
        """Counts the number of nodes in a tree"""
        count = 1  # Count the current node
        for child in tree.children:
            count += self.count_nodes(child)
        return count


In [6]:
table1_html = '''
<html>
  <body>
    <table>
      <tr>
        <td>Header 1</td>
        <td>Header 2</td>
      </tr>
      <tr>
        <td>Data 1</td>
        <td>Data 2</td>
      </tr>
      <tr>
        <td>Data 3</td>
        <td>Data 4</td>
      </tr>
    </table>
  </body>
</html>

'''

table2_html = '''
<html>
  <body>
    <table>
      <tr>
        <td></td>
        <td>Column 1</td>
        <td>Column 2</td>
      </tr>
      <tr>
        <td>Row 1</td>
        <td>Data 1</td>
        <td>Data 2</td>
      </tr>
      <tr>
        <td>Row 2</td>
        <td>Data 3</td>
        <td>Data 4</td>
      </tr>
    </table>
  </body>
</html>

'''

table3_html = '''
<html>
  <body>
    <table>
      <tr>
        <td colspan="2">Header</td>
      </tr>
      <tr>
        <td>Subheader 1</td>
        <td>Subheader 2</td>
      </tr>
      <tr>
        <td>Data 1</td>
        <td>Data 2</td>
      </tr>
    </table>
  </body>
</html>
'''

table4_html = '''
<html>
  <body>
    <table>
      <tr>
        <td>Header 1</td>
        <td>Header 2</td>
      </tr>
      <tr>
        <td>Data 1</td>
        <td rowspan="2">Data 2</td>
      </tr>
      <tr>
        <td>Data 3</td>
        <!-- No <td> here since "Data 2" spans this row -->
      </tr>
      <tr>
        <td>Data 4</td>
        <td></td>
      </tr>
    </table>
  </body>
</html>

'''


table5_html = '''
<html>
  <body>
    <table>
      <tr>
        <td colspan="3">Header 1</td>
        <td>Header 2</td>
      </tr>
      <tr>
        <td colspan="2">Subheader 1</td>
        <td>Subheader 3</td>
      </tr>
      <tr>
        <td>Data 1</td>
        <td rowspan="2">Data 2</td>
        <td>Data 3</td>
      </tr>
      <tr>
        <td>Data 4</td>
        <!-- "Data 2" spans into this row -->
        <td>Data 5</td>
      </tr>
    </table>
  </body>
</html>

'''

table1_json = {
  "type": "table",
  "value": "Sample Table 1",
  "children": [
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Header 1",
        },
        {
          "type": "cell",
          "value": "Header 2",
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 1"
        },
        {
          "type": "cell",
          "value": "Data 2"
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 3"
        },
        {
          "type": "cell",
          "value": "Data 4"
        }
      ]
    }
  ]
}


table2_json = {
  "type": "table",
  "value": "Sample Table 2",
  "children": [
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "",
        },
        {
          "type": "cell",
          "value": "Column 1",
        },
        {
          "type": "cell",
          "value": "Column 2",
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Row 1",
        },
        {
          "type": "cell",
          "value": "Data 1"
        },
        {
          "type": "cell",
          "value": "Data 2"
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Row 2",
        },
        {
          "type": "cell",
          "value": "Data 3"
        },
        {
          "type": "cell",
          "value": "Data 4"
        }
      ]
    }
  ]
}




table3_json = {
  "type": "table",
  "value": "Sample Table 3",
  "children": [
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Header",
          "colspan": 2
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Subheader 1",
        },
        {
          "type": "cell",
          "value": "Subheader 2",
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 1"
        },
        {
          "type": "cell",
          "value": "Data 2"
        }
      ]
    }
  ]
}



table4_json = {
  "type": "table",
  "value": "Sample Table 4",
  "children": [
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Header 1",
        },
        {
          "type": "cell",
          "value": "Header 2",
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 1"
        },
        {
          "type": "cell",
          "value": "Data 2",
          "rowspan": 2
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 3"
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 4"
        },
        {
          "type": "cell",
          "value": ""
        }
      ]
    }
  ]
}



table5_json = {
  "type": "table",
  "value": "Sample Table 5",
  "children": [
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Header 1",
          "colspan": 3
        },
        {
          "type": "cell",
          "value": "Header 2",
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Subheader 1",
          "colspan": 2
        },
        {
          "type": "cell",
          "value": "Subheader 3",
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 1"
        },
        {
          "type": "cell",
          "value": "Data 2",
          "rowspan": 2
        },
        {
          "type": "cell",
          "value": "Data 3"
        }
      ]
    },
    {
      "type": "row",
      "value": "",
      "children": [
        {
          "type": "cell",
          "value": "Data 4"
        },
        {
          "type": "cell",
          "value": "Data 5"
        }
      ]
    }
  ]
}


In [7]:
teds_html = TEDS_HTML(structure_only=False)

gt_html = table1_html
pred_html = table2_html

score = teds_html(gt_html, pred_html)
print(score)


teds_json = TEDS_JSON(structure_only=False)

gt_json = table1_json
pred_json = table2_json

score = teds_json(gt_json, pred_json)
print(score)

0.625
0.625
